In [1]:
import torch
from torch_geometric.data import Data, Dataset
from torch_geometric.data.data import DataEdgeAttr, DataTensorAttr, GlobalStorage

from pathlib import Path
import math
from pyproj import Transformer
from sumolib.net import readNet

/home/marta/tesi-5t/sumo-stgcn/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def calculate_edge_length(shape):
    length = 0.0
    # shape = {(x0, y0), (x1, y1), ... , (xn, yn)} --> lista di tuple
    # iterative eucledian distance computation to get the total lenght
    for i in range(1, len(shape)):
        x1, y1 = shape[i - 1]
        x2, y2 = shape[i]
        length += math.sqrt((x1 - x2) ** 2 + (y1 - y2) ** 2)
    return length

In [112]:
net_file = "/home/marta/tesi-5t/sumo-stgcn/data/raw/francia_peschiera_passenger.net.xml"
net = readNet(
        net_file, withPrograms=True, withConnections=True, withInternal=True
    )

edge_id_to_idx = {}
edge_static_features = []
idx = 0

for edge in net.getEdges():
    if edge.getID().startswith(":"):
        continue
    edge_id_to_idx[edge.getID()] = idx
    idx += 1
    # edge attributes
    lanes = edge.getLanes()
    length = calculate_edge_length(edge.getShape())
    speed_limit = edge.getSpeed()
    num_lanes = len(lanes)
    priority = edge.getPriority()
    capacity = ...
    
    # tensor index = edge id - idx transformation
    # tensor elements = edge attributes 
    edge_static_features.append([length, speed_limit, float(num_lanes), float(priority)])
x_static = torch.tensor(edge_static_features, dtype=torch.long) # shape = [N_elements X N_features]
    
x_static.shape
# len(edge_id_to_idx)

torch.Size([2530, 4])

In [113]:
JUNCTION_TYPE_MAP = {
    "traffic_light": 0,
    "priority": 1,
    "right_before_left": 2,
    "unregulated": 3,
    "dead_end": 4,
    "unknown": 5,
    "internal": 6,
}

In [114]:
TRANSFORMER = Transformer.from_crs("EPSG:32632", "EPSG:4326", always_xy=True)

def _transform_coord(x: float, y:float):
    return TRANSFORMER.transform(x, y)

In [115]:
source_nodes, dest_nodes, attribute_list = [], [], []
seen_conn = set()

for node in net.getNodes():
    incoming = [e for e in node.getIncoming() if not e.getID().startswith(":")]
    outgoing = [e for e in node.getOutgoing() if not e.getID().startswith(":")]
    
    # print(incoming)
    jtype = JUNCTION_TYPE_MAP.get(node.getType(), 5)
    lon, lat = _transform_coord(node.getCoord()[0], node.getCoord()[1])
    has_tls = float(node.getType == "traffic_light")
    tl_link_idx = ...
    
    for e_in in incoming:
        for e_out in outgoing:
            source_id = edge_id_to_idx.get(e_in.getID())
            dest_id = edge_id_to_idx.get(e_out.getID())
            if source_id is None or dest_id is None or source_id == dest_id:
                continue
            connection_pair = (source_id, dest_id)
            if connection_pair in seen_conn:
                continue
            else:
                seen_conn.add(connection_pair)
                source_nodes.append(source_id)
                dest_nodes.append(dest_id)
                attribute_list.append([float(jtype), lon, lat, has_tls])

# source_nodes = where does the connection start
# dest_nodes = where does the connection sink                
connections_idx = torch.tensor([source_nodes, dest_nodes], dtype= torch.long)
edge_attrib = torch.tensor(attribute_list, dtype=torch.long)

connections_idx.shape # shape -> torch.Size([2, 10584])

torch.Size([2, 5292])

In [116]:
edge_attr = torch.tensor(attribute_list, dtype=torch.float)
edge_attr # shape = torch.Size([5292, 4])

tensor([[ 1.0000,  7.6161, 45.0663,  0.0000],
        [ 1.0000,  7.6161, 45.0663,  0.0000],
        [ 1.0000,  7.6161, 45.0661,  0.0000],
        ...,
        [ 4.0000,  7.6146, 45.0778,  0.0000],
        [ 4.0000,  7.6146, 45.0786,  0.0000],
        [ 4.0000,  7.6165, 45.0755,  0.0000]])

In [117]:
FEATURES_MAP = {"flow": 0,
                "speed": 1,
                "occupancy": 2
                }
N_FEATURES = len(FEATURES_MAP)

In [118]:
det_dir = "/home/marta/tesi-5t/sumo-stgcn/data/raw/DetOut_Morning"
n_nodes = len(edge_id_to_idx)

In [125]:
import os
import xml.etree.ElementTree as ET
from collections import defaultdict
from typing import Dict, List

eid = "-36936414"

path = os.path.join(det_dir, f"det_{eid}.xml")
accumulator = defaultdict(lambda: defaultdict(list))
"""
accumulates values from each lane to aggregate records in a second step
{
"begin": {
    "flow": ["val1", "val2", ...],
    "speed": ["val1", "val2", ...],
    "occupancy": ["val1", "val2", ...]
},
...
}
"""
tree = ET.parse(path)
root = tree.getroot()

for interval in root.findall("interval"):
    begin = float(interval.get("begin"))
    fl = float(interval.get("flow"))
    speed = float(interval.get("speed"))
    occ = float(interval.get("occupancy"))
    
    # accumulator takes a dict with all detectors features for each timestep
    accumulator[begin]["flow"].append(float(fl)) if fl is not None else 0.0
    accumulator[begin]["speed"].append(float(speed)) if speed is not None else 0.0
    accumulator[begin]["occupancy"].append(float(occ)) if occ is not None else 0.0
    
accumulator[8*3600]

defaultdict(list, {'flow': [0.0], 'speed': [-1.0], 'occupancy': [0.0]})

In [128]:
records = {}
for begin, lane_data in accumulator.items():
    # each accum dict entry is an edge --> aggregate duplicated value
    feats: Dict[str, float] = {}
    feats['flow'] = sum(lane_data['flow']) # total flow over the street
    feats['speed'] = sum(lane_data['speed']) / len(lane_data['speed'])
    feats['occupancy'] = max(lane_data['occupancy']) # worst case
    records[begin] = feats

records[8*3600]

{'flow': 0.0, 'speed': -1.0, 'occupancy': 0.0}

In [132]:
def _parse_single(eid: str, det_dir: str) -> Dict:
    path = os.path.join(det_dir, f"det_{eid}.xml")
    if not os.path.isfile(path):
        return {}
    
    accumulator = defaultdict(lambda: defaultdict(list))
    """
    accumulates values from each lane to aggregate records in a second step
    {
    "begin": {
        "flow": ["val1", "val2", ...],
        "speed": ["val1", "val2", ...],
        "occupancy": ["val1", "val2", ...]
    },
    ...
    }
    """
    tree = ET.parse(path)
    root = tree.getroot()
    
    for interval in root.findall("interval"):
        begin = float(interval.get("begin"))
        fl = float(interval.get("flow"))
        speed = float(interval.get("speed", -1))
        occ = float(interval.get("occupancy"))
        
        # accumulator takes a dict with all detectors features for each timestep
        accumulator[begin]["flow"].append(float(fl)) if fl is not None else 0.0
        if speed > 0:
            accumulator[begin]["speed"].append(float(speed))
        accumulator[begin]["occupancy"].append(float(occ)) if occ is not None else 0.0
        
    # aggregate by edge 
    records: Dict[float, Dict[str, float]] = {} # record = Dict['begin', features]
    for begin, lane_data in accumulator.items():
        # each accum dict entry is an edge --> aggregate duplicated value
        feats: Dict[str, float] = {}
        feats['flow'] = sum(lane_data['flow']) # total flow over the street
        feats['speed'] = sum(lane_data['speed']) / len(lane_data['speed']) if lane_data['speed'] else 0.0
        feats['occupancy'] = max(lane_data['occupancy']) # worst case
        records[begin] = feats
        
    return records

In [ ]:
def parse(det_dir, edge_id_to_idx) -> torch.Tesnor:
    raw_data = {}
    for eid in edge_id_to_idx:
        records = _parse_single(eid=eid, det_dir=det_dir) # record = Dict['begin', features]
        if records:
            raw_data[eid] = records
            
    if not raw_data:
        raise ValueError("Raw data is empty: check the input data.")

    all_begins = sorted(set(
        float(begin) for records in raw_data.values() for begin in records.keys()
    ), key=float)

            
    # build tensor [Time, Det_edge, Features]
    duration = len(all_begins)
    n_nodes = len(edge_id_to_idx)
    
    begin_to_idx = {b: i for i, b in enumerate(all_begins)}
    
    matrix = torch.full(size=[duration, n_nodes, N_FEATURES], fill_value=0.0, dtype=torch.float32)
    
    # fill matrix
    for det, rec in raw_data.items():
        node_idx = edge_id_to_idx[det]
        for begin, feats in rec.items():
            t = begin_to_idx[begin]
            for fname, fidx in FEATURES_MAP.items():
                if fname in feats:
                    matrix[t, node_idx, fidx] = feats[fname]
        
    return matrix


In [134]:
parse(det_dir, edge_id_to_idx)

edge con dati: 2530 / 2530
esempio edge: -104183437
  timestep trovati: 27
  primo begin: 28800.0
all_begins: 27 timestep


tensor([[[ 0.0000,  0.0000,  0.0000],
         [ 0.0000,  0.0000,  0.0000],
         [ 0.0000,  0.0000,  0.0000],
         ...,
         [24.0000, 13.9300,  0.2400],
         [36.0000, 14.2900,  0.3500],
         [ 0.0000,  0.0000,  0.0000]],

        [[ 0.0000,  0.0000,  0.0000],
         [ 0.0000,  0.0000,  0.0000],
         [ 0.0000,  0.0000,  0.0000],
         ...,
         [12.0000, 14.5200,  0.1100],
         [12.0000, 13.6900,  0.1200],
         [ 0.0000,  0.0000,  0.0000]],

        [[ 0.0000,  0.0000,  0.0000],
         [ 0.0000,  0.0000,  0.0000],
         [ 0.0000,  0.0000,  0.0000],
         ...,
         [ 0.0000,  0.0000,  0.0000],
         [ 0.0000,  0.0000,  0.0000],
         [ 0.0000,  0.0000,  0.0000]],

        ...,

        [[ 0.0000,  0.0000,  0.0000],
         [ 0.0000,  0.0000,  0.0000],
         [ 0.0000,  0.0000,  0.0000],
         ...,
         [ 0.0000,  0.0000,  0.0000],
         [ 0.0000,  0.0000,  0.0000],
         [ 0.0000,  0.0000,  0.0000]],

        [[

In [135]:
torch.load()

TypeError: load() missing 1 required positional argument: 'f'